# Customer demo: local videos to Fabric results

Run **locally in VS Code**, not in a Fabric kernel. Select this repository's `.venv` interpreter.

1. Discover videos and review camera configuration.
2. Calculate SHA-256 identities, recording timestamps, dimensions and duration.
3. Prepare and validate manifests offline.
4. Check the **legacy F64** capacity, Azure CLI storage RBAC, OneLake access and existing automation.
5. Upload videos to **staging** and stop for inspection.
6. Promote videos, then final manifests, to **incoming**.
7. Observe automatic intake/SJD execution and wait for **these exact assets' committed results**.

**All cloud gates are off by default.** Run All is an offline rehearsal. No job is manually submitted by this notebook. Missing automation is a failure, never a simulated success.

Install from a terminal: `uv sync --extra publisher --extra experiments --extra demo`. Also install `ffprobe` and authenticate with `az login`. Do not paste credentials here. Resuming F64 starts billing; enabling recurring schedules has ongoing effects. This notebook does not automatically pause shared capacity or disable schedules.

See [the presenter guide](customer_end_to_end_demo.md) for checkpoints and limitations.

In [35]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import re
import shutil
import time
from dataclasses import replace
from IPython.display import display, clear_output
import pandas as pd

REPO_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'pyproject.toml').is_file())
SOURCE_DIR = REPO_ROOT / 'samples'  # Change to the folder where you paste videos.
CAMERA_CATALOG_CSV = None  # Real catalog: prefixes are relative to SOURCE_DIR.
CAPTURE_OVERRIDES_CSV = None  # Optional authoritative video_relative_path,captured_at_utc overrides.
RUN_LABEL = ''  # Paste the previously printed label here to resume; use a new label for a new arrival.
RUN_LABEL = RUN_LABEL or datetime.now(timezone.utc).strftime('demo-%Y%m%dT%H%M%S%fZ')
if not re.fullmatch('[A-Za-z0-9][A-Za-z0-9_-]*', RUN_LABEL):
    raise ValueError('RUN_LABEL must contain only letters, digits, hyphens and underscores')

SUBSCRIPTION_ID = 'feadfbc6-69f8-42af-b466-e5f36f484fa8'
CAPACITY_RESOURCE_ID = f'/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/rg-people-counting/providers/Microsoft.Fabric/capacities/peoplecounterdev'
CAPACITY_FABRIC_ID = '27fe2dc5-cdfd-4a28-8bba-75413296d02d'
WORKSPACE_ID = 'c31ee864-230d-4005-8fd5-7c7130ebf774'
LAKEHOUSE_ID = '883cff91-eaa8-40be-870f-6e9716303cb2'
ENVIRONMENT_ID = '3e580f48-9ff7-4bc6-af2e-a59158029ada'
STORAGE_ACCOUNT = 'peoplecountingfootage'
FILESYSTEM = 'videos'
STORAGE_RESOURCE_ID = f'/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/rg-people-counting/providers/Microsoft.Storage/storageAccounts/{STORAGE_ACCOUNT}'
SHORTCUT_NAME = 'videos'
TABLE_DIRECTORY = 'Tables/dbo'  # Verified legacy Lakehouse schema directory.

RUN_PREFLIGHT = True
ALLOW_CAPACITY_RESUME = True
ALLOW_ROLE_GRANT = True
ENABLE_EXISTING_SCHEDULES = True  # Only the existing Dispatcher and Gold schedules.
UPLOAD_TO_STAGING = True
PROMOTE_TO_INCOMING = True
MONITOR_FABRIC = True
POLL_SECONDS = 30
MONITOR_TIMEOUT_SECONDS = 7200
MAX_FILES = 10

DEMO_ROOT = REPO_ROOT / '.people-counter-demo'
RUN_DIR = DEMO_ROOT / RUN_LABEL
PACKAGE_DIR = RUN_DIR / 'package'
RUN_DIR.mkdir(parents=True, exist_ok=True)
PREFLIGHT_OK = False
print('Run label (keep this to resume):', RUN_LABEL)
print('Local evidence:', RUN_DIR)
print('Target:', STORAGE_ACCOUNT, FILESYSTEM, 'people-counter-dev-legacy / F64')

Run label (keep this to resume): demo-20261030T173326150299Z
Local evidence: /home/vinny/repos/people-counter/.people-counter-demo/demo-20261030T173326150299Z
Target: peoplecountingfootage videos people-counter-dev-legacy / F64


## 1. Discover videos and review camera metadata

For the shipped samples, each clip is treated as a **different demonstration camera**. Dimensions and embedded timestamps are measured. Centre counting lines and camera/location IDs are **illustrative, not calibrated customer metadata**. The timestamp for `three_people_walking.mp4` is a labelled tutorial override, not a measured recording time.

For customer files, provide `CAMERA_CATALOG_CSV` with the existing operator-tutorial schema: source prefix, camera/location IDs, IANA timezone, dimensions, directed counting line, timestamp rule and effective UTC dates. Missing trustworthy capture times require an authoritative override; filesystem/upload time is never substituted.

A run-specific local copy preserves original bytes but gives each rehearsal new path-based asset IDs. Originals are never renamed or modified. Keep the same run label and copies to resume.

In [36]:
from people_counter.manifest import CONTENT_TYPES, inspect_video

SOURCE_DIR = Path(SOURCE_DIR).resolve()
using_samples = SOURCE_DIR == (REPO_ROOT / 'samples').resolve() and CAMERA_CATALOG_CSV is None
if not using_samples and CAMERA_CATALOG_CSV is None:
    raise ValueError('Provide a real CAMERA_CATALOG_CSV for customer files; sample metadata is not inferred')
source_files = sorted(p for p in SOURCE_DIR.rglob('*') if p.is_file() and p.suffix.lower() in CONTENT_TYPES)
if not 1 <= len(source_files) <= MAX_FILES:
    raise ValueError(f'Expected 1..{MAX_FILES} videos, found {len(source_files)}')
catalog_fields = ['catalog_version', 'source_path_prefix', 'camera_id', 'location_id', 'camera_timezone', 'frame_width', 'frame_height', 'counting_line_x1', 'counting_line_y1', 'counting_line_x2', 'counting_line_y2', 'capture_time_source', 'capture_time_regex', 'capture_time_format', 'effective_from_utc', 'effective_to_utc']
catalog_rows, overrides, previews = [], [], []
snapshot_prefix = f'{RUN_LABEL}/videos/'
for source in source_files:
    if source.is_symlink() or not source.resolve().is_relative_to(SOURCE_DIR):
        raise ValueError(f'Source must be a regular file inside SOURCE_DIR: {source}')
    relative = Path(source.stem) / source.name if using_samples else source.relative_to(SOURCE_DIR)
    destination = DEMO_ROOT / snapshot_prefix / relative
    measured = inspect_video(source)
    if destination.exists():
        if inspect_video(destination).sha256 != measured.sha256:
            raise ValueError('Source bytes changed within this run. Use a new RUN_LABEL.')
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
    previews.append({'file': str(relative), 'width': measured.width, 'height': measured.height, 'seconds': measured.duration_seconds, 'bytes': measured.size_bytes, 'sha256': measured.sha256, 'embedded_times': measured.embedded_capture_times})
    if using_samples:
        camera_prefix = snapshot_prefix + source.stem + '/'
        catalog_rows.append(dict(zip(catalog_fields, [1, camera_prefix, 'DEMO-' + source.stem, 'DEMO-location', 'UTC', measured.width, measured.height, 0, measured.height // 2, measured.width - 1, measured.height // 2, 'auto', '', '', '2000-01-01T00:00:00Z', ''])))
        if not measured.embedded_capture_times:
            if source.name != 'three_people_walking.mp4' or measured.sha256 != '0e70fed4fcdc59334f16dc1eac1a5f433403af4fb53a348a26d24b07404b58d4':
                raise ValueError('No reviewed sample capture-time override for this file')
            overrides.append({'video_relative_path': snapshot_prefix + relative.as_posix(), 'captured_at_utc': '2023-10-01T01:00:00Z'})
if not using_samples:
    with Path(CAMERA_CATALOG_CSV).open(newline='', encoding='utf-8') as handle:
        catalog_rows = list(csv.DictReader(handle))
    for row in catalog_rows:
        row['source_path_prefix'] = snapshot_prefix + row['source_path_prefix']
if CAPTURE_OVERRIDES_CSV is not None:
    with Path(CAPTURE_OVERRIDES_CSV).open(newline='', encoding='utf-8') as handle:
        for row in csv.DictReader(handle):
            overrides.append({'video_relative_path': snapshot_prefix + row['video_relative_path'], 'captured_at_utc': row['captured_at_utc']})
CATALOG_PATH = RUN_DIR / 'camera-catalog.csv'
INVENTORY_PATH = RUN_DIR / 'capture-overrides.csv'
with CATALOG_PATH.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=catalog_fields)
    writer.writeheader()
    writer.writerows(catalog_rows)
with INVENTORY_PATH.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['video_relative_path', 'captured_at_utc'])
    writer.writeheader()
    writer.writerows(overrides)
display(pd.DataFrame(previews))
display(pd.DataFrame(catalog_rows))
display(pd.DataFrame(overrides))

,file,width,height,seconds,bytes,sha256,embedded_times
0,people_crossing/people_crossing.mp4,3626,1994,68.466667,211622638,4a780417943542a7d0e72d4f91f30751b8fad304f91193...,"(2023-09-26T12:18:09.000000Z, 2023-09-26T12:18..."
1,subway/subway.mp4,3840,2160,31.396667,98758271,d5bd760178a4ae0d429a52c2c46ae6ec567fea2d55608e...,"(2023-09-29T16:03:17.000000Z, 2023-09-29T16:03..."
2,three_people_walking/three_people_walking.mp4,1920,1080,7.000000,3458119,0e70fed4fcdc59334f16dc1eac1a5f433403af4fb53a34...,()


,catalog_version,source_path_prefix,camera_id,location_id,camera_timezone,frame_width,frame_height,counting_line_x1,counting_line_y1,counting_line_x2,counting_line_y2,capture_time_source,capture_time_regex,capture_time_format,effective_from_utc,effective_to_utc
0,1,demo-20261030T173326150299Z/videos/people_cros...,DEMO-people_crossing,DEMO-location,UTC,3626,1994,0,997,3625,997,auto,,,2000-01-01T00:00:00Z,
1,1,demo-20261030T173326150299Z/videos/subway/,DEMO-subway,DEMO-location,UTC,3840,2160,0,1080,3839,1080,auto,,,2000-01-01T00:00:00Z,
2,1,demo-20261030T173326150299Z/videos/three_peopl...,DEMO-three_people_walking,DEMO-location,UTC,1920,1080,0,540,1919,540,auto,,,2000-01-01T00:00:00Z,


,video_relative_path,captured_at_utc
0,demo-20261030T173326150299Z/videos/three_peopl...,2023-10-01T01:00:00Z


## 2. Prepare and verify the checksum package (offline)

Reuse the production preparation and validation code. `asset_id = SHA256(relative path)`; `asset_version = SHA256(exact video bytes)`. The camera catalog and prepared JSON files are checksum-indexed. Rejections stop the demo with remediation reports.

Prepared manifests deliberately omit storage URI and ETag. Final manifests can be created only after the video has its final incoming ETag. No Azure access occurs in this section.

In [37]:
from people_counter.manifest_publisher import ManifestPackagePublishConfig, PublisherConfig, CheckpointStore, prepare_main, validate_manifest_package
from people_counter.customer_demo import azure_cli, ensure_capacity, ensure_storage_role, stage_videos, promote_staged, asset_progress

prepare_args = ['--catalog', str(CATALOG_PATH), '--video-root', str(DEMO_ROOT), '--partition-prefix', snapshot_prefix, '--output-dir', str(PACKAGE_DIR), '--max-files', str(MAX_FILES)]
if overrides:
    prepare_args += ['--inventory', str(INVENTORY_PATH)]
package_config = ManifestPackagePublishConfig(PACKAGE_DIR, DEMO_ROOT, STORAGE_ACCOUNT, FILESYSTEM, 'staging', 'incoming', RUN_DIR / 'publication.sqlite3', RUN_DIR / 'publication-rejections.csv', RUN_DIR / 'publication-summary.json', 8 * 1024 * 1024)
index_path = PACKAGE_DIR / 'manifest-package.json'
previous_index = json.loads(index_path.read_text()) if index_path.exists() else None
if previous_index is not None and previous_index['complete'] is True:
    inventory_digest = hashlib.sha256(INVENTORY_PATH.read_bytes()).hexdigest() if overrides else None
    if previous_index['catalog_sha256'] != hashlib.sha256(CATALOG_PATH.read_bytes()).hexdigest() or previous_index['inventory_sha256'] != inventory_digest:
        raise RuntimeError('Run metadata changed. Use a new RUN_LABEL rather than replacing a complete package.')
    print('Reusing the complete package; validating checksums and source bytes without regenerating it.')
else:
    if (RUN_DIR / 'arrival-state.json').exists():
        raise RuntimeError('Published run is missing its complete package. Restore the original package before resuming.')
    exit_code = prepare_main(prepare_args)
    if exit_code:
        print((PACKAGE_DIR / 'rejection-report.csv').read_text())
        raise RuntimeError(f'Preparation rejected inputs (exit {exit_code}); repair metadata before continuing')
validated = validate_manifest_package(package_config)
plans = validated.plans
if len(plans) != len(source_files):
    raise RuntimeError('Prepared input count differs from discovered videos')
package_sha256 = hashlib.sha256(validated.index_bytes).hexdigest()
display(pd.read_csv(PACKAGE_DIR / 'generated-video-inventory.csv'))
print('Package SHA-256:', package_sha256)
print('Example prepared manifest:')
print((PACKAGE_DIR / validated.index['entries'][0]['prepared_manifest_path']).read_text())
display(pd.DataFrame([{'file': p.relative_path, 'staging': p.staging_video_path, 'incoming': p.incoming_video_path, 'manifest_published_last': p.incoming_manifest_path} for p in plans]))

{"already_published": 0, "catalog_sha256": "35836e73124fcf2765c01a305096bd6403fff611c1d39ee0723d1014cbe227b8", "discovered": 3, "generator_version": "0.9.57", "inventory_sha256": "ed7393f4aa4b660d1a4565068cfa71fedad8d892a992faea4890f04a28d75350", "planned": 3, "published": 0, "rejected": 0, "rejections_by_camera": {}, "rejections_by_prefix": {}, "rejections_by_reason": {}, "total_video_duration_seconds": 106.86333400000001}


,video_relative_path,camera_id,location_id,camera_timezone,captured_at_utc,capture_time_source,asset_id,asset_version,size_bytes,sha256,frame_width,frame_height,duration_seconds,prepared_manifest_path
0,demo-20261030T173326150299Z/videos/people_cros...,DEMO-people_crossing,DEMO-location,UTC,2023-09-26T12:18:09Z,embedded,221cfb431506d1c9a41d6e004671eb93d5962116b52667...,4a780417943542a7d0e72d4f91f30751b8fad304f91193...,211622638,4a780417943542a7d0e72d4f91f30751b8fad304f91193...,3626,1994,68.466667,prepared-manifests/221cfb431506d1c9a41d6e00467...
1,demo-20261030T173326150299Z/videos/subway/subw...,DEMO-subway,DEMO-location,UTC,2023-09-29T16:03:17Z,embedded,48f2d9a71b2eb2ce0ce2314e0a728ffc9830cee633cff0...,d5bd760178a4ae0d429a52c2c46ae6ec567fea2d55608e...,98758271,d5bd760178a4ae0d429a52c2c46ae6ec567fea2d55608e...,3840,2160,31.396667,prepared-manifests/48f2d9a71b2eb2ce0ce2314e0a7...
2,demo-20261030T173326150299Z/videos/three_peopl...,DEMO-three_people_walking,DEMO-location,UTC,2023-10-01T01:00:00Z,inventory_override,e14221f5d0b1b6f6954d298a57b3be79865624b6b94a43...,0e70fed4fcdc59334f16dc1eac1a5f433403af4fb53a34...,3458119,0e70fed4fcdc59334f16dc1eac1a5f433403af4fb53a34...,1920,1080,7.000000,prepared-manifests/e14221f5d0b1b6f6954d298a57b...


Package SHA-256: b9cf94af697bff28daf6a3f2361c79146dc8e37036ff42701787767069c7fc6a
Example prepared manifest:
{
  "asset_id": "221cfb431506d1c9a41d6e004671eb93d5962116b52667d2811ed3218fed1a25",
  "asset_version": "4a780417943542a7d0e72d4f91f30751b8fad304f911936ce7c07ee5a49584c5",
  "camera_id": "DEMO-people_crossing",
  "camera_timezone": "UTC",
  "capture_time_source": "embedded",
  "captured_at_utc": "2023-09-26T12:18:09Z",
  "content_type": "video/mp4",
  "counting_line": [
    0,
    997,
    3625,
    997
  ],
  "duration_seconds": 68.466667,
  "expected_sha256": "4a780417943542a7d0e72d4f91f30751b8fad304f911936ce7c07ee5a49584c5",
  "expected_size_bytes": 211622638,
  "frame_height": 1994,
  "frame_width": 3626,
  "location_id": "DEMO-location",
  "prepared_manifest_version": 1,
  "relative_video_path": "demo-20261030T173326150299Z/videos/people_crossing/people_crossing.mp4",
  "source_path_prefix": "demo-20261030T173326150299Z/videos/people_crossing/"
}



,file,staging,incoming,manifest_published_last
0,demo-20261030T173326150299Z/videos/people_cros...,staging/221cfb431506d1c9a41d6e004671eb93d59621...,incoming/2023/09/26/221cfb431506d1c9a41d6e0046...,incoming/2023/09/26/221cfb431506d1c9a41d6e0046...
1,demo-20261030T173326150299Z/videos/subway/subw...,staging/48f2d9a71b2eb2ce0ce2314e0a728ffc9830ce...,incoming/2023/09/29/48f2d9a71b2eb2ce0ce2314e0a...,incoming/2023/09/29/48f2d9a71b2eb2ce0ce2314e0a...
2,demo-20261030T173326150299Z/videos/three_peopl...,staging/e14221f5d0b1b6f6954d298a57b3be79865624...,incoming/2023/10/01/e14221f5d0b1b6f6954d298a57...,incoming/2023/10/01/e14221f5d0b1b6f6954d298a57...


## 3. Live preflight: F64, the CLI identity, and existing automation

Review the target above. Set `RUN_PREFLIGHT=True`. If paused, `ALLOW_CAPACITY_RESUME=True` permits a resume and waits for **Active**. This starts F64 billing; the identity needs capacity read/resume permission (for example a capacity-scoped Contributor assignment).

The notebook uses **AzureCliCredential explicitly**, not an environment service principal. It checks inherited/group storage data roles. `ALLOW_ROLE_GRANT=True` permits only **Storage Blob Data Contributor on this container**, never subscription-wide Owner. Assigning this role requires Owner, User Access Administrator or an equivalent role-assignment permission. Custom roles/ACL-only setups need operator review. A bounded retry performs a real staging write/read/delete probe after RBAC propagation; Azure management Contributor alone is not data access.

Fabric workspace/OneLake access is separate from Azure RBAC. The CLI identity also needs suitable Fabric workspace/item read permissions, and permission to update schedules only if enabling them. Missing rights stop the demo; the notebook never grants Fabric workspace Admin automatically.

Legacy Dispatcher processes batches **inside its own SJD run**. A separate Process SJD instance is therefore not required. Dispatcher and Gold must have existing schedules enabled. `ENABLE_EXISTING_SCHEDULES=True` enables only those existing schedules, preserving their configuration; it does not invent a registration trigger or launch jobs.

**Arrival delivery gate:** the notebook requires the selected storage-event source, stream, and destination to the arrival Activator to report `Running`, with no runtime error. It also verifies that `run_pc_event_intake_on_manifest_renamed` is enabled and targets the selected intake pipeline. These checks repeat before staging upload, before incoming promotion, and on every monitoring poll.

**If paused:** open `azure_storage_events_eventstream` in Fabric **Live view**, activate `to_pc_manifest_arrival_activator`, and review the **When streaming was last stopped** resume option for retained events. If the rule is disabled, start it in Activator. Then rerun preflight using the same run label. The notebook verifies the public runtime status; it does not call an undocumented activation endpoint. An inactive path blocks publication immediately instead of waiting two hours. Already-published manifests must not be deleted or re-uploaded to manufacture an event; verify backlog delivery after activation.

In [38]:
if RUN_PREFLIGHT:
    PREFLIGHT_OK = False
    from azure.identity import AzureCliCredential
    from azure.core.exceptions import HttpResponseError
    from azure.storage.filedatalake import DataLakeServiceClient
    from people_counter.adls import AzureDataLakeStorage
    from people_counter.fabric_automation import FabricAutomationClient, AzureCliResourceTokenProvider
    from people_counter.fabric_rest import UrllibTransport
    import duckdb
    from importlib import reload
    import people_counter.customer_demo as demo_helpers
    read_demo_delta = reload(demo_helpers).read_demo_delta
    import people_counter.fabric_sjd_definition as definition_helpers
    reload(definition_helpers)

    capacity = azure_cli(['resource', 'show', '--ids', CAPACITY_RESOURCE_ID, '--api-version', '2023-11-01'])
    if capacity['sku']['name'] != 'F64':
        raise RuntimeError('Expected the reviewed legacy F64; refuse a different capacity')
    account = azure_cli(['account', 'show'])
    if account['id'] != SUBSCRIPTION_ID:
        raise RuntimeError('Wrong CLI subscription; select the reviewed subscription before continuing')
    print('Capacity:', ensure_capacity(CAPACITY_RESOURCE_ID, allow_resume=ALLOW_CAPACITY_RESUME))
    if account['user']['type'] == 'user':
        principal_type = 'User'
        principal_id = azure_cli(['ad', 'signed-in-user', 'show', '--query', 'id'])
    elif account['user']['type'] == 'servicePrincipal':
        principal_type = 'ServicePrincipal'
        principal_id = azure_cli(['ad', 'sp', 'show', '--id', account['user']['name'], '--query', 'id'])
    else:
        raise RuntimeError('Use an attended Azure CLI user or explicitly reviewed service-principal login')
    ensure_storage_role(STORAGE_RESOURCE_ID, FILESYSTEM, principal_id, principal_type, allow_grant=ALLOW_ROLE_GRANT)
    credential = AzureCliCredential()
    storage = AzureDataLakeStorage(STORAGE_ACCOUNT, FILESYSTEM, chunk_size=package_config.chunk_size, credential=credential)
    filesystem_client = DataLakeServiceClient(f'https://{STORAGE_ACCOUNT}.dfs.core.windows.net', credential=credential).get_file_system_client(FILESYSTEM)
    probe_path = f'staging/demo-probes/{RUN_LABEL}-{time.time_ns()}.json'
    access_deadline = time.monotonic() + 300
    while True:
        try:
            storage.upload_bytes(b'{"demo_access_probe":true}', probe_path, 'application/json')
            try:
                if storage.read_bytes(probe_path) != b'{"demo_access_probe":true}':
                    raise RuntimeError('Storage write/read probe returned different bytes')
            finally:
                filesystem_client.get_file_client(probe_path).delete_file()
            break
        except HttpResponseError as error:
            if error.status_code != 403 or time.monotonic() >= access_deadline:
                raise
            print('Storage data-plane access not effective yet; retrying RBAC propagation in 15 seconds')
            time.sleep(15)
    print('CLI identity: container write/read/delete probe passed')
    fabric = FabricAutomationClient(AzureCliResourceTokenProvider('https://api.fabric.microsoft.com'), transport=UrllibTransport(), trusted_redirect_host_suffixes=('.analysis.windows.net',))
    workspace = fabric.request_json('GET', f'/workspaces/{WORKSPACE_ID}')
    if workspace['capacityId'] != CAPACITY_FABRIC_ID:
        raise RuntimeError('Workspace is not attached to the reviewed legacy F64')
    items = fabric.list_pages(f'/workspaces/{WORKSPACE_ID}/items')
    item_names = {'intake': 'pc-event-intake', 'arrival': 'pc_manifest_arrival_activator', 'eventstream': 'azure_storage_events_eventstream', 'control': 'pc-sjd-production-control-v001', 'dispatcher': 'pc-sjd-dispatcher', 'process': 'pc-sjd-production-process-v001', 'gold': 'pc-sjd-production-gold-v001'}
    item_ids = {}
    for role, name in item_names.items():
        matches = [item for item in items if item['displayName'] == name]
        if len(matches) != 1:
            raise RuntimeError(f'Expected exactly one existing {name}; found {len(matches)}')
        item_ids[role] = matches[0]['id']

    def check_arrival_runtime():
        demo_helpers.verify_release_alignment(fabric, WORKSPACE_ID, item_ids['intake'], item_ids['dispatcher'])
        demo_helpers.verify_compute_policy(fabric, WORKSPACE_ID, ENVIRONMENT_ID)
        workspace_pool = fabric.request_json('GET', f'/workspaces/{WORKSPACE_ID}/spark/settings')['pool']
        desired_workers = int(next(row['value'] for row in definition_helpers.reviewed_compute_settings({})['sparkProperties'] if row['key'] == 'spark.executor.instances'))
        if workspace_pool.get('customizeComputeEnabled') is not True or workspace_pool['starterPool'].get('maxExecutors') != desired_workers or workspace_pool['starterPool'].get('maxNodeCount') != desired_workers + 1:
            raise RuntimeError('Workspace Starter Pool limits differ from the reviewed allocation; published Environment properties alone are not sufficient. Ask an approved workspace admin to correct them before continuing.')
        return demo_helpers.verify_arrival_runtime(fabric, workspace_id=WORKSPACE_ID, eventstream_id=item_ids['eventstream'], activator_id=item_ids['arrival'], intake_id=item_ids['intake'], storage_resource_id=STORAGE_RESOURCE_ID)

    print('Arrival readiness:', check_arrival_runtime())
    shortcut = fabric.request_json('GET', f'/workspaces/{WORKSPACE_ID}/items/{LAKEHOUSE_ID}/shortcuts/Files/{SHORTCUT_NAME}')
    adls_target = shortcut['target']['adlsGen2']
    if adls_target['location'].rstrip('/') != f'https://{STORAGE_ACCOUNT}.dfs.core.windows.net' or adls_target['subpath'].strip('/') != FILESYSTEM:
        raise RuntimeError('Lakehouse shortcut does not point to the selected storage container root')
    for role in ('dispatcher', 'gold'):
        path = f'/workspaces/{WORKSPACE_ID}/items/{item_ids[role]}/jobs/sparkjob/schedules'
        schedules = fabric.list_pages(path)
        if len(schedules) != 1:
            raise RuntimeError(f'{role}: expected one existing schedule; configure it in Fabric before the demo')
        schedule = schedules[0]
        if not schedule['enabled']:
            if not ENABLE_EXISTING_SCHEDULES:
                raise RuntimeError(f'{role} schedule is disabled. Review it, then approve ENABLE_EXISTING_SCHEDULES or enable it in Fabric.')
            fabric.request_json('PATCH', path + '/' + schedule['id'], value={'enabled': True, 'configuration': schedule['configuration']}, expected=(200, 202))
            if not fabric.list_pages(path)[0]['enabled']:
                raise RuntimeError(f'{role} schedule enablement is not yet visible; rerun preflight')
        print(role, 'schedule enabled')

    delta_reader = duckdb.connect()
    delta_reader.execute("INSTALL azure; LOAD azure; INSTALL delta; LOAD delta; SET azure_transport_option_type='curl';")
    delta_reader.execute("CREATE SECRET onelake (TYPE azure, PROVIDER credential_chain, CHAIN 'cli', ACCOUNT_NAME 'onelake', ENDPOINT 'https://onelake.blob.fabric.microsoft.com')")
    table_prefix_uri = f'abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/{TABLE_DIRECTORY}/people_counter_sjd_'
    onelake_files = DataLakeServiceClient('https://onelake.dfs.fabric.microsoft.com', credential=credential).get_file_system_client(WORKSPACE_ID)
    diagnostic_cache = {}

    def check_processing_failure(work_ids, started_at):
        if not work_ids:
            return
        for role in ('dispatcher', 'process'):
            directory = f'{LAKEHOUSE_ID}/Files/people-counter/sjd/v1/{role}/diagnostics'
            if not onelake_files.get_directory_client(directory).exists():
                continue
            for path in onelake_files.get_paths(path=directory, recursive=False):
                timestamp = re.search(r'-(\d+)-[0-9a-f]+\.json$', path.name)
                if timestamp is not None and int(timestamp.group(1)) >= int(started_at):
                    if path.name not in diagnostic_cache:
                        diagnostic_cache[path.name] = json.loads(onelake_files.get_file_client(path.name).download_file().readall())
        demo_helpers.raise_processing_failure(list(diagnostic_cache.values()), work_ids, started_at=started_at)

    def read_delta(suffix, columns, filters=None):
        return read_demo_delta(delta_reader, table_prefix_uri, suffix, columns, filters)

    read_delta('work', ['work_id', 'payload_json', 'status', 'created_at', 'committed_attempt_id'], [('created_at', '>=', time.time())])
    read_delta('gold_video', ['work_id', 'distinct_people'], [('work_id', 'in', ['demo-preflight-no-match'])])
    read_delta('gold_work_operations', ['work_id', 'committed_attempt_id'], [('work_id', 'in', ['demo-preflight-no-match'])])
    PREFLIGHT_OK = True
    print('Preflight passed. Item presence does not prove trigger execution; monitoring below requires fresh asset-specific work and new job instances.')
    print(f'Arrival rule: https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}/reflexes/{item_ids["arrival"]}')
else:
    print('Live preflight skipped (offline rehearsal). No capacity, RBAC, schedule or storage changes.')

Capacity: Active
CLI identity: container write/read/delete probe passed
Arrival readiness: {'source_status': 'Running', 'stream_status': 'Running', 'destination_status': 'Running', 'rule_enabled': True}
dispatcher schedule enabled
gold schedule enabled
Preflight passed. Item presence does not prove trigger execution; monitoring below requires fresh asset-specific work and new job instances.
Arrival rule: https://app.fabric.microsoft.com/groups/c31ee864-230d-4005-8fd5-7c7130ebf774/reflexes/c39f8c1d-e363-402d-b7f0-34f53ce31bcc


## 4. Upload to staging; do not trigger intake yet

Set `UPLOAD_TO_STAGING=True` after preflight. Every remote staged video is read back and SHA-256 checked. Conflicting staging data and existing incoming paths are refused, not overwritten.

**Pause here for the customer:** the files are complete in staging, but no final incoming manifest exists. The local prepared manifest is reviewable. Final storage-dependent manifests will be staged during promotion after the video's final ETag is known.

In [39]:
if UPLOAD_TO_STAGING:
    if not PREFLIGHT_OK:
        raise RuntimeError('Run a successful live preflight first')
    plans = validate_manifest_package(package_config).plans
    check_arrival_runtime()
    stage_videos(storage, plans)
    display(pd.DataFrame([{'file': p.relative_path, 'staged_path': p.staging_video_path, 'bytes': storage.stat(p.staging_video_path).size, 'sha256_verified': True} for p in plans]))
    print('STOP: staging verified. Set UPLOAD_TO_STAGING=False before proceeding.')
else:
    print('Staging upload disabled.')

,file,staged_path,bytes,sha256_verified
0,demo-20261030T173326150299Z/videos/people_cros...,staging/221cfb431506d1c9a41d6e004671eb93d59621...,211622638,True
1,demo-20261030T173326150299Z/videos/subway/subw...,staging/48f2d9a71b2eb2ce0ce2314e0a728ffc9830ce...,98758271,True
2,demo-20261030T173326150299Z/videos/three_peopl...,staging/e14221f5d0b1b6f6954d298a57b3be79865624...,3458119,True


STOP: staging verified. Set UPLOAD_TO_STAGING=False before proceeding.


## 5. Promote staging to incoming: the arrival boundary

Set `PROMOTE_TO_INCOMING=True` only when ready to start downstream work. This cell records a durable pre-arrival job baseline and timestamp **before moving anything**, then reuses checkpointed production publication logic. Each video moves first; its final manifest is created in staging and moved **last**.

If interrupted, keep the same run label and rerun this cell; never delete existing incoming data to force a trigger. Do not rerun the staging cell after promotion. New rehearsals need a new run label. Registration/Spark startup may take minutes.

In [40]:
STATE_PATH = RUN_DIR / 'arrival-state.json'
if PROMOTE_TO_INCOMING:
    if not PREFLIGHT_OK:
        raise RuntimeError('Run a successful live preflight first')
    check_arrival_runtime()
    validated = validate_manifest_package(package_config)
    plans = validated.plans
    binding = {'workspace': WORKSPACE_ID, 'lakehouse': LAKEHOUSE_ID, 'account': STORAGE_ACCOUNT, 'filesystem': FILESYSTEM, 'package_sha256': hashlib.sha256(validated.index_bytes).hexdigest()}
    if STATE_PATH.exists():
        arrival = json.loads(STATE_PATH.read_text())
        if arrival['binding'] != binding:
            raise RuntimeError('Targets or package changed since arrival; do not reuse this run state')
    else:
        baseline = {role: [job['id'] for job in fabric.list_pages(f'/workspaces/{WORKSPACE_ID}/items/{item_ids[role]}/jobs/instances')] for role in ('intake', 'control', 'dispatcher', 'process', 'gold')}
        arrival = {'binding': binding, 'started_at': time.time(), 'baseline_jobs': baseline}
        with STATE_PATH.open('x', encoding='utf-8') as handle:
            json.dump(arrival, handle, indent=2)
    publish_config = PublisherConfig(CATALOG_PATH, INVENTORY_PATH if overrides else None, DEMO_ROOT, snapshot_prefix, STORAGE_ACCOUNT, FILESYSTEM, 'staging', 'incoming', package_config.checkpoint_path, package_config.rejection_report_path, PACKAGE_DIR / 'generated-video-inventory.csv', MAX_FILES, False)
    checkpoint = CheckpointStore(package_config.checkpoint_path)
    try:
        published = promote_staged(storage, checkpoint, publish_config, plans)
    finally:
        checkpoint.close()
    print(f'New manifests published: {published}; remaining entries were verified on retry.')
    for plan in plans:
        print(plan.incoming_manifest_path)
    print('Publication boundary complete. Set PROMOTE_TO_INCOMING=False, then enable monitoring.')
else:
    print('Incoming promotion disabled.')

New manifests published: 3; remaining entries were verified on retry.
incoming/2023/09/26/221cfb431506d1c9a41d6e004671eb93d5962116b52667d2811ed3218fed1a25/4a780417943542a7d0e72d4f91f30751b8fad304f911936ce7c07ee5a49584c5/people_crossing.json
incoming/2023/09/29/48f2d9a71b2eb2ce0ce2314e0a728ffc9830cee633cff0284bd6de365b56c3dc/d5bd760178a4ae0d429a52c2c46ae6ec567fea2d55608e982293bfc001a8f750/subway.json
incoming/2023/10/01/e14221f5d0b1b6f6954d298a57b3be79865624b6b94a431af549da512e5e89d6/0e70fed4fcdc59334f16dc1eac1a5f433403af4fb53a348a26d24b07404b58d4/three_people_walking.json
Publication boundary complete. Set PROMOTE_TO_INCOMING=False, then enable monitoring.


## 6. Observe automatic execution and these assets' results

Set `MONITOR_FABRIC=True`. The notebook **does not submit intake, registration, claim, processing or Gold jobs**. It compares paginated job histories with the pre-arrival baseline, and independently matches stable work by **exact source path + content SHA-256 + registration time**. It requires the committed attempt pointer to match the Gold operational projection before showing the Gold video result as complete.

Completion requires all input results plus a new intake pipeline run and a new Dispatcher or Process SJD run. New job timestamps alone never count as processing proof. Job histories are contextual evidence, not a one-to-one job-to-asset mapping or independent proof of Activator causality; the stable work/attempt pointers prove asset execution. No old result or unrelated successful Spark run can satisfy the asset checks.

A timeout is a failed demo checkpoint. Inspect the arrival Activator, Eventstream, event-intake pipeline, Control registration payload, Dispatcher/Gold schedules and job failure details. Existing legacy intake might target an older queue; this notebook will not silently switch to that queue or manually register inputs to conceal the gap.

In [41]:
if MONITOR_FABRIC:
    if not PREFLIGHT_OK or not STATE_PATH.exists():
        raise RuntimeError('Successful preflight and durable incoming-promotion state are required')
    arrival = json.loads(STATE_PATH.read_text())
    current_package = validate_manifest_package(package_config)
    expected_binding = {'workspace': WORKSPACE_ID, 'lakehouse': LAKEHOUSE_ID, 'account': STORAGE_ACCOUNT, 'filesystem': FILESYSTEM, 'package_sha256': hashlib.sha256(current_package.index_bytes).hexdigest()}
    if arrival['binding'] != expected_binding:
        raise RuntimeError('Monitoring target/package differs from the publication state')
    plans = current_package.plans
    deadline = time.monotonic() + MONITOR_TIMEOUT_SECONDS
    while True:
        check_arrival_runtime()
        work_rows = read_delta('work', ['work_id', 'payload_json', 'status', 'created_at', 'committed_attempt_id', 'last_error'], [('created_at', '>=', arrival['started_at'])])
        progress = asset_progress(plans, work_rows, [], started_at=arrival['started_at'], shortcut=SHORTCUT_NAME)
        ids = [row['work_id'] for row in progress if 'work_id' in row]
        check_processing_failure(ids, arrival['started_at'])
        result_rows = []
        if ids:
            gold = read_delta('gold_video', ['work_id', 'camera_id', 'location_id', 'video_duration_seconds', 'processing_seconds', 'speed_x_realtime', 'distinct_people', 'line_in_count', 'line_out_count', 'completed_at'], [('work_id', 'in', ids)])
            operations = read_delta('gold_work_operations', ['work_id', 'committed_attempt_id'], [('work_id', 'in', ids)])
            pointers = {row['work_id']: row['committed_attempt_id'] for row in operations}
            result_rows = [dict(row, attempt_id=pointers.get(row['work_id'])) for row in gold]
        progress = asset_progress(plans, work_rows, result_rows, started_at=arrival['started_at'], shortcut=SHORTCUT_NAME)
        jobs = []
        for role, baseline in arrival['baseline_jobs'].items():
            for job in fabric.list_pages(f'/workspaces/{WORKSPACE_ID}/items/{item_ids[role]}/jobs/instances'):
                if job['id'] not in baseline:
                    jobs.append({'role': role, 'job_id': job['id'], 'status': job['status'], 'invoke_type': job.get('invokeType'), 'start_time': job.get('startTimeUtc'), 'failure': job.get('failureReason')})
        clear_output(wait=True)
        print('Run:', RUN_LABEL, '| Poll:', datetime.now(timezone.utc).isoformat())
        display(pd.DataFrame(progress))
        display(pd.DataFrame(jobs))
        evidence = {'run_label': RUN_LABEL, 'arrival': arrival, 'observed_at': datetime.now(timezone.utc).isoformat(), 'assets': progress, 'new_jobs': jobs}
        evidence_path = RUN_DIR / 'execution-evidence.json'
        temporary = evidence_path.with_suffix('.tmp')
        temporary.write_text(json.dumps(evidence, indent=2, default=str), encoding='utf-8')
        temporary.replace(evidence_path)
        observed_roles = {job['role'] for job in jobs if job['status'] == 'Completed'}
        complete = all(row['state'] == 'COMPLETED' for row in progress)
        if complete and 'intake' in observed_roles and observed_roles.intersection({'dispatcher', 'process'}):
            print('VERIFIED: every uploaded asset has a fresh registration, committed attempt, and Gold result; new intake and processing SJD activity observed.')
            print('Persistent evidence:', evidence_path)
            break
        if time.monotonic() >= deadline:
            raise TimeoutError('No complete automatic execution evidence before timeout. Inspect execution-evidence.json, Activator/intake wiring, source_video mapping, and Dispatcher/Gold failures. No manual jobs were launched.')
        time.sleep(POLL_SECONDS)
else:
    print('Fabric monitoring disabled. Offline rehearsal complete; no live execution is claimed.')

Run: demo-20261030T173326150299Z | Poll: 2026-10-30T18:58:18.672309+00:00


,file,asset_id,sha256,state,work_id,work_status,committed_attempt_id
0,demo-20261030T173326150299Z/videos/people_cros...,221cfb431506d1c9a41d6e004671eb93d5962116b52667...,4a780417943542a7d0e72d4f91f30751b8fad304f91193...,LEASED,f8971d8ab546c9ade424456f2a18dadc9cf13b5d981717...,LEASED,None
1,demo-20261030T173326150299Z/videos/subway/subw...,48f2d9a71b2eb2ce0ce2314e0a728ffc9830cee633cff0...,d5bd760178a4ae0d429a52c2c46ae6ec567fea2d55608e...,LEASED,e86002c80bc96fe3c579d9e7de74243e0f5b18ef6bd46e...,LEASED,None
2,demo-20261030T173326150299Z/videos/three_peopl...,e14221f5d0b1b6f6954d298a57b3be79865624b6b94a43...,0e70fed4fcdc59334f16dc1eac1a5f433403af4fb53a34...,LEASED,5199154441573a9b62d58a2f85c59ec4392035712e5377...,LEASED,None


,role,job_id,status,invoke_type,start_time,failure
0,intake,e9269892-119e-476a-bf38-a3def857b2d5,Completed,Manual,2026-10-30T17:36:45.5146057,None
1,intake,9de3fa1e-61fd-4c75-a211-a9ff8f52a896,Completed,Manual,2026-10-30T17:36:44.3505098,None
2,intake,27bea46c-4268-4d6e-ab09-4c9c6a4add8f,Completed,Manual,2026-10-30T17:36:29.2452961,None
3,control,a6a4c5e3-7201-4105-9170-f2d10d6cd3be,Completed,Manual,2026-10-30T17:39:11.6512244,None
4,control,69dbcead-81c0-40c2-9c33-46b8123e1e29,Completed,Manual,2026-10-30T17:39:11.1829028,None
5,control,ee1a32c1-b911-4003-bb42-5fc587880965,Completed,Manual,2026-10-30T17:39:11.1458552,None
6,dispatcher,65c0d4e0-40c0-4da7-928c-df06c6c589b2,InProgress,Scheduled,2026-10-30T18:55:03.7270113,None
7,dispatcher,f4295452-8f8e-42e2-ae66-f454c18bbec2,Completed,Scheduled,2026-10-30T18:50:03.6459351,None
8,dispatcher,42591867-7287-4b1a-8fc1-f3eb3c1a0eff,Completed,Scheduled,2026-10-30T18:45:06.6200368,None
9,dispatcher,cd83860a-8e84-41f9-956a-24b49e14acb8,Completed,Scheduled,2026-10-30T18:40:05.5440636,None


KeyboardInterrupt: 

## Presenter close and cost control

Show the per-video duration, processing seconds, distinct people, and exact work/committed attempt IDs. Counting lines are demonstration values: successful inference does **not** guarantee nonzero entry/exit counts. Physical Gold rows are read directly; a Power BI refresh is not claimed by this notebook.

Keep the run folder and evidence to resume or explain failures. Do not commit notebook outputs or local evidence containing customer data. Original files and immutable incoming data are retained.

**F64 stays running and recurring schedules stay enabled if you enabled them.** After all dependent jobs finish, coordinate with other users before pausing the shared capacity or disabling schedules. To pause, an authorized operator can use Azure Portal or the documented Fabric capacity `suspend` API. Pausing during active work can interrupt processing. This notebook does not perform destructive cleanup.